In [49]:
import requests, base64

In [50]:
invoke_url = "https://ai.api.nvidia.com/v1/vlm/microsoft/phi-3-vision-128k-instruct"

In [51]:
with open("../img/demo.png", "rb") as f:
  image_b64 = base64.b64encode(f.read()).decode()

In [52]:
len(image_b64)

118688

In [53]:
assert len(image_b64) < 180_000, \
  "To upload larger images, use the assets API (see docs)"

In [54]:
stream = False

In [55]:
import os

In [56]:
headers = {
  "Authorization": os.getenv("NVIDIA_API_KEY"),
  "Accept": "text/event-stream" if stream else "application/json"
}


In [57]:
# payload = {
#   "messages": [
#     {
#       "role": "user",
#       "content": f'Tell me what is in the picture <img src="data:image/png;base64,{image_b64}" />'
#     }
#   ],
#   "max_tokens": 1024,
#   "temperature": 0.6,
#   "top_p": 1.0,
#   "stream": stream
# }

payload = {
  "messages": [
    {
      "role": "user",
      "content": f'Please create Python code for image, and use plt to save the new picture under imgs/ and name it phi-3-vision.jpg. <img src="data:image/png;base64,{image_b64}" />'
    }
  ],
  "max_tokens": 1024,
  "temperature": 0.6,
  "top_p": 1.0,
  "stream": stream
}

In [58]:
response = requests.post(invoke_url, headers=headers, json=payload)

In [59]:
if stream:
    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))
else:
    print(response.json())

{'id': '0094005f-d98c-4cbb-84cf-9b0ec12e04ce', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "To create Python code for a given image, we can use libraries such as OpenCV and matplotlib. Here is a sample code that can be used to accomplish this task:\n\n```python\nimport cv2\nimport matplotlib.pyplot as plt\n\n# Load the image\nimg = cv2.imread('image.jpg')\n\n# Convert the image to grayscale\ngray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n\n# Apply Gaussian blur\nblurred = cv2.GaussianBlur(gray, (5, 5), 0)\n\n# Apply Canny edge detection\nedges = cv2.Canny(blurred, 50, 150)\n\n# Display the result\nplt.imshow(edges, cmap='gray')\nplt.axis('off')\nplt.savefig('imgs/phi-3-vision.jpg')\n```\n\nThis code reads an image, converts it to grayscale, applies Gaussian blur and Canny edge detection, and then displays the resulting image. Finally, it saves the image as `imgs/phi-3-vision.jpg`.\n\nNote: Make sure to replace `'image.jpg'` with the path to the image you want to

In [60]:
resp_json = response.json()
if "choices" in resp_json:
	code = resp_json["choices"][0]["message"]["content"]
else:
	print("Error in response:", resp_json)
	code = None

In [61]:
code

"To create Python code for a given image, we can use libraries such as OpenCV and matplotlib. Here is a sample code that can be used to accomplish this task:\n\n```python\nimport cv2\nimport matplotlib.pyplot as plt\n\n# Load the image\nimg = cv2.imread('image.jpg')\n\n# Convert the image to grayscale\ngray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n\n# Apply Gaussian blur\nblurred = cv2.GaussianBlur(gray, (5, 5), 0)\n\n# Apply Canny edge detection\nedges = cv2.Canny(blurred, 50, 150)\n\n# Display the result\nplt.imshow(edges, cmap='gray')\nplt.axis('off')\nplt.savefig('imgs/phi-3-vision.jpg')\n```\n\nThis code reads an image, converts it to grayscale, applies Gaussian blur and Canny edge detection, and then displays the resulting image. Finally, it saves the image as `imgs/phi-3-vision.jpg`.\n\nNote: Make sure to replace `'image.jpg'` with the path to the image you want to process."

In [62]:
# Find the starting index of the Python code block within the generated code
# The index function locates the substring '```python' and adds 9 to skip past it
begin = code.index('```python') + 9

# Slice the code string from the calculated starting index to the end
# This removes the initial part of the string up to and including '```python'
code = code[begin:]

In [63]:
end = code.index('```')

In [64]:
code = code[:end]

In [65]:
code

"\nimport cv2\nimport matplotlib.pyplot as plt\n\n# Load the image\nimg = cv2.imread('image.jpg')\n\n# Convert the image to grayscale\ngray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n\n# Apply Gaussian blur\nblurred = cv2.GaussianBlur(gray, (5, 5), 0)\n\n# Apply Canny edge detection\nedges = cv2.Canny(blurred, 50, 150)\n\n# Display the result\nplt.imshow(edges, cmap='gray')\nplt.axis('off')\nplt.savefig('imgs/phi-3-vision.jpg')\n"

In [66]:
clean_code = code.encode().decode('unicode_escape')
print("```python\n" + clean_code + "\n```")


```python

import cv2
import matplotlib.pyplot as plt

# Load the image
img = cv2.imread('image.jpg')

# Convert the image to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Apply Gaussian blur
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Apply Canny edge detection
edges = cv2.Canny(blurred, 50, 150)

# Display the result
plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.savefig('imgs/phi-3-vision.jpg')

```
